# Классификация тарифов: Smart vs Ultra

Строим модель классификации для предсказания тарифа (`is_ultra`) и добиваемся `accuracy >= 0.75`.
Используем ансамблевые методы и подбираем гиперпараметры для `RandomForestClassifier`.

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, GridSearchCV

In [2]:
RANDOM_STATE = 12345
DATA_PATH = 'users_behavior.csv'
TARGET = 'is_ultra'

data = pd.read_csv(DATA_PATH)
X = data.drop(columns=[TARGET])
y = data[TARGET]

# 60% train, 20% valid, 20% test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=RANDOM_STATE, stratify=y_train_full
)

print(f'Train size: {len(X_train)}')
print(f'Valid size: {len(X_valid)}')
print(f'Test size: {len(X_test)}')

Train size: 1928
Valid size: 643
Test size: 643


In [3]:
# Подбор гиперпараметров для RandomForest
rf = RandomForestClassifier(random_state=RANDOM_STATE)
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 8, 12, 16],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_search = GridSearchCV(
    rf,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)
rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_

print('Best RF params:', rf_search.best_params_)
print('Best CV score:', round(rf_search.best_score_, 4))

Best RF params: {'max_depth': 8, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 200}
Best CV score: 0.806


In [4]:
# Обучение ансамблей и сравнение на validation
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
et = ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)

models = {
    'RandomForest (tuned)': best_rf,
    'GradientBoosting': gb,
    'ExtraTrees': et
}

valid_scores = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred_valid = model.predict(X_valid)
    score = accuracy_score(y_valid, pred_valid)
    valid_scores[name] = score
    print(f'{name}: validation accuracy = {score:.4f}')

voting = VotingClassifier(
    estimators=[('rf', best_rf), ('gb', gb), ('et', et)],
    voting='hard'
)
voting.fit(X_train, y_train)
voting_valid = accuracy_score(y_valid, voting.predict(X_valid))
valid_scores['VotingClassifier'] = voting_valid
print(f'VotingClassifier: validation accuracy = {voting_valid:.4f}')

best_name = max(valid_scores, key=valid_scores.get)
print('Best model on validation:', best_name)

RandomForest (tuned): validation accuracy = 0.8243
GradientBoosting: validation accuracy = 0.8149
ExtraTrees: validation accuracy = 0.8087
VotingClassifier: validation accuracy = 0.8180
Best model on validation: RandomForest (tuned)


In [5]:
# Финальная проверка на тесте
final_model = voting if best_name == 'VotingClassifier' else models[best_name]
final_model.fit(X_train_full, y_train_full)

test_pred = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_pred)

print(f'Final test accuracy ({best_name}) = {test_accuracy:.4f}')
print('Target reached:', test_accuracy >= 0.75)

Final test accuracy (RandomForest (tuned)) = 0.8134
Target reached: True


Если `Target reached: True`, требование задачи выполнено.
При необходимости можно расширить `param_grid` или добавить `RandomizedSearchCV`.

## Выводы по лабораторной работе №6

В рамках лабораторной работы была решена задача бинарной классификации тарифов (`Smart`/`Ultra`) по пользовательскому поведению (`is_ultra`).

Что было сделано:
- Загружен и подготовлен датасет `users_behavior.csv`, выделены признаки и целевая переменная.
- Выполнено разбиение данных с сохранением баланса классов: 60% — обучение, 20% — валидация, 20% — тест.
- Проведён подбор гиперпараметров `RandomForestClassifier` через `GridSearchCV` (метрика — `accuracy`, `cv=5`).
- Обучены и сравнены несколько ансамблевых моделей: `RandomForest`, `GradientBoosting`, `ExtraTrees`, а также `VotingClassifier`.
- Выбрана лучшая модель по качеству на валидационной выборке и выполнена финальная проверка на тестовой выборке.

Итоговые результаты:
- Лучшая модель на валидации — **настроенный `RandomForestClassifier`** (`accuracy = 0.8243`).
- На тестовой выборке получено **`accuracy = 0.8134`**.
- Требование задачи (**`accuracy >= 0.75`**) выполнено.

Общий вывод: цель лабораторной достигнута — построена устойчивая модель классификации тарифов с качеством выше порогового значения. Наиболее эффективным подходом в данной задаче оказался `RandomForest` после подбора гиперпараметров.